# RECOLECCIÓN DE PARÁMETROS INICIALES

In [0]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo destino")
catalog = dbutils.widgets.get("catalog")
schema_silver = dbutils.widgets.get("schema_silver")

spark.sql(f"USE CATALOG {catalog}")
print(f"Catálogo activo: {catalog}")

# CREACIÓN DE ESQUEMAS

In [0]:
esquemas = {
    "bronze": "Capa de ingesta cruda (raw data)",
    "silver": "Capa de datos limpios, conformados e historial SCD Type 2",
    "gold": "Capa analítica y reglas de negocio"
}

for nombre, descripcion in esquemas.items():
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {catalog}.{nombre}
        COMMENT '{descripcion}'
    """)
    print(f"Esquema verificado/creado: {catalog}.{nombre}")

# CREACIÓN DE VOLUMES

In [0]:
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {catalog}.default.vol_landing
    COMMENT 'Volume landing para archivos crudos de Electrocasa'
""")

carpetas = [
    "catalogo",
    "devoluciones",
    "empleados",
    "resenas",
    "sucursales",
    "ventas",
    "tracking_envios"
]

for carpeta in carpetas:
    dbutils.fs.mkdirs(f"/Volumes/{catalog}/default/vol_landing/{carpeta}")

print(f"Volume y carpetas listos en: /Volumes/{catalog}/default/vol_landing/")

# GOBIERNO DE DATOS

In [0]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.{schema_silver}.mask_dni(dni STRING)
    RETURNS STRING
    RETURN CASE
        WHEN is_account_group_member('electrocasa_ingenieria') THEN dni
        ELSE CONCAT('****', RIGHT(dni, 4))
    END
""")

spark.sql(f"""
    CREATE OR REPLACE FUNCTION {catalog}.{schema_silver}.mask_salario(salario DECIMAL(10, 2))
    RETURNS DECIMAL(10, 2)
    RETURN CASE
        WHEN is_account_group_member('electrocasa_ingenieria') THEN salario
        ELSE NULL
    END
""")